In [1]:
from osgeo import gdal
print(gdal.VersionInfo("--version"))
print(gdal.GetDriverByName("HDF4"))
print(gdal.GetDriverByName("HDF4Image"))
print(gdal.GetDriverByName("HDF4_EOS"))

GDAL 3.12.4 "Chicoutimi", released 2026/04/22
<osgeo.gdal.Driver; proxy of <Swig Object of type 'GDALDriverShadow *' at 0x14e912ac7e30> >
<osgeo.gdal.Driver; proxy of <Swig Object of type 'GDALDriverShadow *' at 0x14e912aedcb0> >
None


In [2]:
import os
import re
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone

import geopandas as gpd
import numpy as np
import rasterio
import xarray as xr
from rasterio.mask import mask

# MOD10A1.061 snow grid and science / QA fields (HDF-EOS)
MOD10_GRID = "MOD_Grid_Snow_500m"
MOD10_SNOW_FIELD = "NDSI_Snow_Cover"
MOD10_BASIC_QA_FIELD = "NDSI_Snow_Cover_Basic_QA"

In [3]:
def _local_tag(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag


def read_modis_hdf_files(directory: str, pattern: str = "MOD10A1") -> list[str]:
    """Return sorted paths to MODIS granule HDF4 files (not XML)."""
    paths = []
    for f in os.listdir(directory):
        if not f.endswith(".hdf"):
            continue
        if pattern not in f:
            continue
        paths.append(os.path.join(directory, f))
    return sorted(paths)


def granule_xml_path(hdf_path: str) -> str | None:
    """NSIDC-style sidecar metadata: same basename + .xml (often .hdf.xml)."""
    for candidate in (hdf_path + ".xml", hdf_path.replace(".hdf", ".hdf.xml")):
        if os.path.isfile(candidate):
            return candidate
    return None


def parse_granule_datetime_from_xml(xml_path: str) -> datetime | None:
    """Best-effort RangeBeginningDateTime / SingleDateTime from EOS metadata XML."""
    try:
        root = ET.parse(xml_path).getroot()
        want = {
            "RangeBeginningDateTime",
            "SingleDateTime",
            "BeginningDateTime",
            "RangeEndingDateTime",
        }
        for el in root.iter():
            if _local_tag(el.tag) in want and el.text:
                txt = el.text.strip()
                if not txt:
                    continue
                if txt.endswith("Z"):
                    txt = txt[:-1] + "+00:00"
                return datetime.fromisoformat(txt)
        return None
    except Exception:
        return None


def extract_date_from_modis_filename(hdf_path: str) -> datetime:
    """
    Parse MOD10A1.AYYYYDDD.* filename convention (collection date = YYYY + day-of-year).
    """
    base = os.path.basename(hdf_path)
    m = re.search(r"\.A(\d{4})(\d{3})\.", base)
    if not m:
        raise ValueError(f"Cannot parse MODIS granule date from filename: {base}")
    year, doy = int(m.group(1)), int(m.group(2))
    return datetime(year, 1, 1, tzinfo=timezone.utc) + timedelta(days=doy - 1)


def granule_date_for_hdf(hdf_path: str) -> datetime:
    """
    Granule calendar date (UTC midnight for the MODIS day-of-year), preferring XML if present.
    Returns timezone-aware UTC datetimes for CF time axis.
    """
    xml_p = granule_xml_path(hdf_path)
    d_xml = parse_granule_datetime_from_xml(xml_p) if xml_p else None
    d_fn = extract_date_from_modis_filename(hdf_path)
    if d_xml:
        if d_xml.tzinfo is None:
            d_xml = d_xml.replace(tzinfo=timezone.utc)
        if abs((d_xml.date() - d_fn.date()).days) > 1:
            print(
                f"Warning: XML vs filename date differ for {os.path.basename(hdf_path)}: "
                f"{d_xml.date()} vs {d_fn.date()} — using XML."
            )
        return d_xml
    return d_fn


def _hdf4_eos_uri_variants(uri: str) -> list[str]:
    """Quoted vs unquoted path in HDF4_EOS:EOS_GRID — rasterio often needs unquoted."""
    out: list[str] = []
    if uri:
        out.append(uri)
    m = re.match(r'^HDF4_EOS:EOS_GRID:"([^"]+)"(:.*)$', uri)
    if m:
        alt = f"HDF4_EOS:EOS_GRID:{m.group(1)}{m.group(2)}"
        if alt not in out:
            out.append(alt)
    else:
        m2 = re.match(r"^HDF4_EOS:EOS_GRID:(/?.+\.hdf)(:.*)$", uri, re.IGNORECASE)
        if m2:
            alt2 = f'HDF4_EOS:EOS_GRID:"{m2.group(1)}"{m2.group(2)}'
            if alt2 not in out:
                out.append(alt2)
    return out


def _rasterio_resolved_subdataset_uri(uri: str) -> str | None:
    for v in _hdf4_eos_uri_variants(uri):
        try:
            with rasterio.open(v) as src:
                if src.width > 0 and src.height > 0:
                    return v
        except Exception:
            continue
    return None


def _collect_gdal_subdataset_names(hdf_path: str) -> list[str]:
    """Subdataset connection strings as GDAL reports them (authoritative for HDF4/HDF5 EOS)."""
    names: list[str] = []
    try:
        with rasterio.open(hdf_path) as src:
            names.extend(list(src.subdatasets or []))
    except Exception:
        pass
    try:
        from osgeo import gdal

        gdal.UseExceptions()
        ds = gdal.Open(hdf_path, gdal.GA_ReadOnly)
        if ds is not None:
            names.extend([pair[0] for pair in ds.GetSubDatasets()])
    except Exception:
        pass
    seen: set[str] = set()
    out: list[str] = []
    for n in names:
        if n and n not in seen:
            seen.add(n)
            out.append(n)
    return out


def _fallback_subdataset_uris(hdf_path: str, grid: str, field: str) -> list[str]:
    """Try common GDAL spelling variants (quoted vs unquoted path; HDF5 Data_Fields path)."""
    hdf_path = os.path.abspath(os.path.expanduser(hdf_path))
    base = os.path.basename(hdf_path)
    uris: list[str] = []
    for pathpart in (hdf_path, base):
        uris.append(f'HDF4_EOS:EOS_GRID:"{pathpart}":{grid}:{field}')
        uris.append(f"HDF4_EOS:EOS_GRID:{pathpart}:{grid}:{field}")
    uris.append(f'HDF5:"{hdf_path}"://{grid}/Data_Fields/{field}')
    uris.append(f"HDF5:{hdf_path}://{grid}/Data_Fields/{field}")
    seen: set[str] = set()
    out: list[str] = []
    for u in uris:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


def resolve_hdf_subdataset(hdf_path: str, grid: str, field: str) -> str:
    """
    Return a rasterio-openable subdataset URI for an HDF-EOS grid field (MOD10A1 etc.).

    GDAL's ``HDF4_EOS:EOS_GRID:`` spelling varies by build; always prefer names from
    ``GetSubDatasets()`` / ``src.subdatasets``, then probe fallbacks.
    """
    hdf_path = os.path.abspath(os.path.expanduser(hdf_path))
    if not os.path.isfile(hdf_path):
        raise FileNotFoundError(f"HDF granule not found: {hdf_path}")
    with open(hdf_path, "rb") as _f:
        _head = _f.read(800)
    if _head.lstrip().startswith((b"<!DOCTYPE", b"<html", b"<?xml")):
        raise ValueError(
            f"File starts like XML/HTML, not binary HDF (check download/auth): {hdf_path!r}"
        )

    names = _collect_gdal_subdataset_names(hdf_path)
    # Prefer exact field and grid match
    ranked: list[tuple[int, str]] = []
    for sd in names:
        if field not in sd:
            continue
        score = 0
        if sd.rstrip('"').endswith(":" + field) or sd.endswith("/" + field):
            score += 3
        if f":{grid}:{field}" in sd:
            score += 2
        elif f"/{grid}/" in sd and field in sd:
            score += 2
        elif grid in sd:
            score += 1
        ranked.append((score, sd))
    ranked.sort(key=lambda x: -x[0])
    for _score, sd in ranked:
        ok = _rasterio_resolved_subdataset_uri(sd)
        if ok:
            return ok

    for cand in _fallback_subdataset_uris(hdf_path, grid, field):
        ok = _rasterio_resolved_subdataset_uri(cand)
        if ok:
            return ok

    # Some GDAL builds resolve EOS_GRID only when the path in the connection string is
    # the basename and the process cwd is the granule directory — try that, then
    # prefer absolute-path variants that still open.
    hr_dir = os.path.dirname(hdf_path)
    base = os.path.basename(hdf_path)
    if hr_dir and base:
        _cwd = os.getcwd()
        try:
            os.chdir(hr_dir)
            for tmpl in (
                f'HDF4_EOS:EOS_GRID:"{base}":{grid}:{field}',
                f"HDF4_EOS:EOS_GRID:{base}:{grid}:{field}",
            ):
                ok = _rasterio_resolved_subdataset_uri(tmpl)
                if not ok:
                    continue
                for full in (
                    f'HDF4_EOS:EOS_GRID:"{hdf_path}":{grid}:{field}',
                    f"HDF4_EOS:EOS_GRID:{hdf_path}:{grid}:{field}",
                ):
                    full_ok = _rasterio_resolved_subdataset_uri(full)
                    if full_ok:
                        return full_ok
                return ok
        except OSError:
            pass
        finally:
            try:
                os.chdir(_cwd)
            except OSError:
                pass

    if ranked and ranked[0][0] >= 3:
        last = _rasterio_resolved_subdataset_uri(ranked[0][1])
        if last:
            return last

    preview = names[:12]
    raise RuntimeError(
        f"Could not open field {grid}:{field} in {hdf_path!r}. "
        "Your GDAL may lack HDF4-EOS (install a gdal build linked to HDF4 on conda-forge), "
        "the granule may be corrupt, or subdataset names may differ from expected.\n"
        f"Subdatasets reported: {len(names)} total; first: {preview!r}\n"
        "Tip: run `gdalinfo <file.hdf>` on the cluster and use a SUBDATASET_…_NAME as the open path."
    )

def _valid_mod10_snow_mask(snow: np.ndarray) -> np.ndarray:
    """Mask: MOD10A1 NDSI_Snow_Cover 0–100 only (excludes Key class codes 200+)."""
    s = np.asarray(snow)
    return (s >= 0) & (s <= 100)


def _valid_mod10_basic_qa_mask(qa: np.ndarray, allowed: tuple[int, ...]) -> np.ndarray:
    """Keep pixels whose Basic QA is in allowed; exclude 255 and other class flags."""
    q = np.asarray(qa)
    return np.isin(q, np.asarray(allowed, dtype=q.dtype))


def calculate_hru_mean_ndsi_snow_cover(
    hdf_path: str,
    hrus: gpd.GeoDataFrame,
    *,
    apply_basic_qa: bool = True,
    allowed_basic_qa: tuple[int, ...] = (0, 1, 2),
) -> list[tuple]:
    """
    Mean NDSI snow cover (0–100) per HRU from MOD10A1 Data_Fields, sinusoidal grid.
    Invalid / flagged pixels are excluded from the mean (NaN if none valid).
    """
    snow_sd = resolve_hdf_subdataset(hdf_path, MOD10_GRID, MOD10_SNOW_FIELD)
    qa_sd = resolve_hdf_subdataset(hdf_path, MOD10_GRID, MOD10_BASIC_QA_FIELD)

    out = []
    with rasterio.open(snow_sd) as snow_src:
        hrus_r = hrus.to_crs(snow_src.crs)
        qa_ds = None
        if apply_basic_qa:
            qa_ds = rasterio.open(qa_sd)
        try:
            for hru in hrus_r.itertuples():
                geom = [hru.geometry]
                snow_arr, _ = mask(snow_src, geom, crop=True, filled=False)
                snow2d = snow_arr[0]
                if np.ma.isMaskedArray(snow2d):
                    snow2d = np.asarray(snow2d.filled(np.uint8(255)))

                if apply_basic_qa and qa_ds is not None:
                    qa_arr, _ = mask(qa_ds, geom, crop=True, filled=False)
                    qa2d = qa_arr[0]
                    if np.ma.isMaskedArray(qa2d):
                        qa2d = np.asarray(qa2d.filled(np.uint8(255)))
                    else:
                        qa2d = np.asarray(qa2d)
                    qa_ok = _valid_mod10_basic_qa_mask(qa2d, allowed_basic_qa)
                else:
                    qa_ok = np.ones(snow2d.shape, dtype=bool)

                valid = _valid_mod10_snow_mask(snow2d) & qa_ok

                # Align QA mask shape if crop windows differ (should not happen for same grid)
                if valid.shape != snow2d.shape:
                    raise ValueError("Snow and QA crop shapes differ; check granule.")

                vals = snow2d[valid]
                if vals.size == 0:
                    out.append((hru.HRU_ID, np.nan))
                else:
                    out.append((hru.HRU_ID, float(np.mean(vals))))
        finally:
            if qa_ds is not None:
                qa_ds.close()

    return out


def granule_calendar_date_utc(hdf_path: str) -> datetime:
    """UTC midnight on the MODIS granule calendar day (for merging h/v tiles)."""
    dt = granule_date_for_hdf(hdf_path)
    d = dt.astimezone(timezone.utc).date()
    return datetime(d.year, d.month, d.day, tzinfo=timezone.utc)


def aggregate_modis_by_hru(hdf_paths: list[str], hrus: gpd.GeoDataFrame, **kwargs) -> dict:
    from collections import defaultdict

    by_hru_day = defaultdict(lambda: defaultdict(list))
    for hdf_path in hdf_paths:
        day = granule_calendar_date_utc(hdf_path)
        means = calculate_hru_mean_ndsi_snow_cover(hdf_path, hrus, **kwargs)
        for hru_id, val in means:
            by_hru_day[hru_id][day].append(val)

    data = {}
    for hru_id, day_map in by_hru_day.items():
        series = []
        for day in sorted(day_map):
            vals = [v for v in day_map[day] if not np.isnan(v)]
            series.append((day, float(np.mean(vals)) if vals else np.nan))
        data[hru_id] = series
    return data

def write_ndsi_snow_cover_netcdf(
    hru_time_data: dict,
    output_file: str,
    *,
    apply_basic_qa: bool,
    allowed_basic_qa: tuple[int, ...],
) -> None:
    """CF-friendly NetCDF: HRU x time mean MOD10A1 NDSI_Snow_Cover (0–100)."""
    hru_keys_sorted = sorted(hru_time_data.keys(), key=lambda k: str(k))
    hru_ids_str = [str(k) for k in hru_keys_sorted]
    times = sorted({t for series in hru_time_data.values() for t, _ in series})

    def _to_np_dt64(dt: datetime) -> np.datetime64:
        u = dt.astimezone(timezone.utc).replace(tzinfo=None)
        return np.datetime64(u.isoformat())

    np_time = np.array([_to_np_dt64(t) for t in times])

    arr = np.full((len(hru_ids_str), len(times)), np.nan, dtype=np.float64)
    for i, k in enumerate(hru_keys_sorted):
        series = hru_time_data[k]
        for t, v in series:
            j = times.index(t)
            arr[i, j] = v

    ds = xr.Dataset(
        {
            "ndsi_snow_cover": (
                ["hru", "time"],
                arr,
                {
                    "long_name": "NDSI snow cover from best observation of the day",
                    "units": "none",
                    "coordinates": "HRU_ID time",
                    "cell_methods": "time: mean (area: mean)",
                    "comment": (
                        "Mean of MOD10A1.061 NDSI_Snow_Cover over HRU; "
                        "valid pixels 0–100 only; MODIS class flags excluded."
                    ),
                },
            )
        },
        coords={
            "HRU_ID": ("hru", hru_ids_str, {"long_name": "Hydrologic Response Unit ID"}),
            "time": (
                "time",
                np_time,
                {"long_name": "Time", "standard_name": "time", "axis": "T"},
            ),
        },
        attrs={
            "title": "MOD10A1 NDSI snow cover aggregated by HRU",
            "summary": (
                "Area-mean NDSI_Snow_Cover (0–100) from MOD10A1.061 over HRU polygons "
                "reprojected to the MODIS sinusoidal grid. Multiple tiles on the same UTC "
                "calendar day: simple mean of per-tile HRU means (not area-weighted)."
            ),
            "Conventions": "CF-1.8",
            "institution": "Your Institution Here",
            "history": f"Created {datetime.now(timezone.utc).isoformat()}",
            "source": "MODIS MOD10A1.061 / NDSI_Snow_Cover",
            "references": "https://doi.org/10.5067/MODIS/MOD10A1.061",
            "identifier_product_doi": "10.5067/MODIS/MOD10A1.061",
            "identifier_product_doi_authority": "http://dx.doi.org",
            "mod10_basic_qa_applied": str(apply_basic_qa),
            "mod10_basic_qa_allowed_values": ",".join(str(x) for x in allowed_basic_qa),
        },
    )

    enc = {"ndsi_snow_cover": {"zlib": True, "complevel": 4, "_FillValue": np.nan}}
    ds.to_netcdf(output_file, format="NETCDF4", encoding=enc)



In [7]:
def main():
    from osgeo import gdal

    # Silence noisy HDF5 probes against HDF4-EOS MODIS files
    gdal.PushErrorHandler("CPLQuietErrorHandler")

    # MOD10A1.061 HDF granules
    directory = "/anvil/projects/x-ees240082/users/dcasson/modis/tuolumne/"
    hrus_file = "/anvil/projects/x-ees240082/users/dcasson/gpep/tuolumne/gis/tuolumne_tdx.gpkg"
    netcdf_output = (
        "/anvil/projects/x-ees240082/users/dcasson/modis/tuolumne_mod10a1_ndsi_snow_cover.nc"
    )

    apply_basic_qa = True
    allowed_basic_qa = (0, 1, 2)  # best, good, ok

    hrus = gpd.read_file(hrus_file)

    hdf_paths = read_modis_hdf_files(directory, pattern="MOD10A1")
    if not hdf_paths:
        print(f"No MOD10A1 .hdf files found in {directory}")
        return

    valid_hdf_paths = []
    for hdf_path in hdf_paths:
        ds = gdal.Open(str(hdf_path), gdal.GA_ReadOnly)
        if ds is None:
            print(f"Skipping unreadable HDF: {hdf_path}")
            continue

        subdataset_names = [name for name, _ in ds.GetSubDatasets()]
        has_snow = any(
            name.endswith(":MOD_Grid_Snow_500m:NDSI_Snow_Cover")
            for name in subdataset_names
        )
        has_qa = any(
            name.endswith(":MOD_Grid_Snow_500m:NDSI_Snow_Cover_Basic_QA")
            for name in subdataset_names
        )

        if not has_snow:
            print(f"Skipping HDF with no NDSI_Snow_Cover subdataset: {hdf_path}")
            continue

        if apply_basic_qa and not has_qa:
            print(f"Skipping HDF with no Basic_QA subdataset: {hdf_path}")
            continue

        valid_hdf_paths.append(hdf_path)

    if not valid_hdf_paths:
        print("No readable MOD10A1 HDF files with required subdatasets found.")
        return

    print(f"Processing {len(valid_hdf_paths)} valid MOD10A1 HDF files")

    hru_data = aggregate_modis_by_hru(
        valid_hdf_paths,
        hrus,
        apply_basic_qa=apply_basic_qa,
        allowed_basic_qa=allowed_basic_qa,
    )

    write_ndsi_snow_cover_netcdf(
        hru_data,
        netcdf_output,
        apply_basic_qa=apply_basic_qa,
        allowed_basic_qa=allowed_basic_qa,
    )

    print(f"Wrote MOD10A1 NDSI_Snow_Cover HRU means to {netcdf_output}")


if __name__ == "__main__":
    main()

Processing 4000 valid MOD10A1 HDF files


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 827 in H5Fopen(): unable to synchronously open file
    major: File accessibility
    minor: Unable to open file
  #001: H5F.c line 788 in H5F__open_api_common(): unable to open file
    major: File accessibility
    minor: Unable to open file
  #002: H5VLcallback.c line 3680 in H5VL_file_open(): open failed
    major: Virtual Object Layer
    minor: Can't open object
  #003: H5VLcallback.c line 3514 in H5VL__file_open(): open failed
    major: Virtual Object Layer
    minor: Can't open object
  #004: H5VLnative_file.c line 128 in H5VL__native_file_open(): unable to open file
    major: File accessibility
    minor: Unable to open file
  #005: H5Fint.c line 2085 in H5F_open(): unable to read superblock
    major: File accessibility
    minor: Read failed
  #006: H5Fsuper.c line 386 in H5F__super_read(): file signature not found
    major: File accessibility
    minor: Not an HDF5 file
HDF5-DIAG: Error detected in H

RuntimeError: Could not open field MOD_Grid_Snow_500m:NDSI_Snow_Cover in '/anvil/projects/x-ees240082/users/dcasson/modis/tuolumne/MOD10A1.A2008275.h08v05.061.2021107213849.hdf'. Your GDAL may lack HDF4-EOS (install a gdal build linked to HDF4 on conda-forge), the granule may be corrupt, or subdataset names may differ from expected.
Subdatasets reported: 7 total; first: ['HDF4_EOS:EOS_GRID:"/anvil/projects/x-ees240082/users/dcasson/modis/tuolumne/MOD10A1.A2008275.h08v05.061.2021107213849.hdf":MOD_Grid_Snow_500m:NDSI_Snow_Cover', 'HDF4_EOS:EOS_GRID:"/anvil/projects/x-ees240082/users/dcasson/modis/tuolumne/MOD10A1.A2008275.h08v05.061.2021107213849.hdf":MOD_Grid_Snow_500m:NDSI_Snow_Cover_Basic_QA', 'HDF4_EOS:EOS_GRID:"/anvil/projects/x-ees240082/users/dcasson/modis/tuolumne/MOD10A1.A2008275.h08v05.061.2021107213849.hdf":MOD_Grid_Snow_500m:NDSI_Snow_Cover_Algorithm_Flags_QA', 'HDF4_EOS:EOS_GRID:"/anvil/projects/x-ees240082/users/dcasson/modis/tuolumne/MOD10A1.A2008275.h08v05.061.2021107213